In [1]:
!pip show transformers

Name: transformers
Version: 4.48.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /home/userloc/miniconda3/envs/nlp/lib/python3.11/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: sentence-transformers, trl


Класс AutoModel и все его представители на самом деле являются простыми обертками для широкого спектра моделей, доступных в библиотеке. Это умная обертка, поскольку она может автоматически определить архитектуру модели, подходящую для вашей контрольной точки, а затем инстанцировать модель с этой архитектурой.

In [64]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id

In [65]:
prompt = "SSH service is"

Перевод текста в числа называется кодированием (encoding)

первым шагом является разбиение текста на слова (или части слов, знаки препинания и т. д.), обычно называемые токенами

In [66]:
tokenizer.tokenize(prompt)

['SSH', 'Ġservice', 'Ġis']

In [36]:
tokenizer.encode(prompt)

[62419, 2473, 374]

Декодирование происходит наоборот: из индексов словаря мы хотим получить строку

In [40]:
decoded_string = tokenizer.decode([62419, 2473, 374])
decoded_string

'SSH service is'

Генерация

In [68]:
prompt = "SSH service is"

inputs = tokenizer([prompt], return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=10)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


['SSH service is a service that provides a secure, reliable, and']

на более низком уровне

In [61]:
import torch

print(prompt, '\n')

voc = tokenizer.get_vocab()
voc_rev = {v:k for k, v in voc.items()}  # reverse vocab for decode

for i in range(10):
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    # next_token_id = torch.multinomial(probs.flatten(), num_samples=1)
    next_token_id = torch.argmax(probs).item()
    next_token = tokenizer.decode(next_token_id)
    prompt += next_token

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    top_tokens = sorted_indices[:5]
    print(f"Step #{i} candidates:")
    for t, p in zip (top_tokens, sorted_probs):
        t = voc_rev[t.item()]
        print(f"{t:<10}: {p:.4f} ")

    print(f'\nChosen token: {next_token}', end='\n\n', flush=True)


SSH service is 

Step #0 candidates:
Ġa        : 0.1416 
Ġused     : 0.0859 
Ġrunning  : 0.0591 
Ġthe      : 0.0459 
Ġconfigured: 0.0405 

Chosen token:  a

Step #1 candidates:
Ġservice  : 0.1250 
Ġset      : 0.0593 
Ġtype     : 0.0522 
Ġsoftware : 0.0317 
Ġkind     : 0.0262 

Chosen token:  service

Step #2 candidates:
Ġthat     : 0.3906 
Ġprovided : 0.1118 
Ġwhich    : 0.0527 
Ġof       : 0.0364 
Ġrunning  : 0.0364 

Chosen token:  that

Step #3 candidates:
Ġprovides : 0.1807 
Ġruns     : 0.1240 
Ġis       : 0.1094 
Ġallows   : 0.0850 
Ġcan      : 0.0752 

Chosen token:  provides

Step #4 candidates:
Ġa        : 0.2031 
Ġservices : 0.0850 
Ġthe      : 0.0659 
Ġremote   : 0.0659 
Ġnetwork  : 0.0454 

Chosen token:  a

Step #5 candidates:
Ġsecure   : 0.0996 
Ġservice  : 0.0532 
Ġremote   : 0.0391 
ĠREST     : 0.0344 
Ġset      : 0.0344 

Chosen token:  secure

Step #6 candidates:
Ġand      : 0.1914 
,         : 0.1914 
Ġconnection: 0.0903 
Ġway      : 0.0801 
Ġcommunication: 0.0427 

C

Стратегии сэмплирования

## Генерация текста

**Сравнение стратегий для генерации текста языковыми моделями:**

| Стратегия | Описание | Плюсы и минусы |
| --- | --- | --- |
| Жадный поиск (Greedy Search) | Выбирает слово с наибольшей вероятностью в качестве следующего слова в последовательности. | **Плюсы:** Простота и скорость. <br> **Минусы:** Может приводить к повторяющемуся и несвязному тексту. |
| Сэмплирование с температурой (Sampling with Temperature) | Вносит случайность в выбор слова. Более высокая температура увеличивает степень случайности. | **Плюсы:** Позволяет исследовать и создавать разнообразные тексты. <br> **Минусы:** Высокая температура может приводить к бессмысленным результатам. |
| Ядерное сэмплирование (Nucleus Sampling, Top-p Sampling) | Выбирает следующее слово из усечённого словаря, "ядра" слов, чья совокупная вероятность превышает заданный порог (p). | **Плюсы:** Баланс между разнообразием и качеством. <br> **Минусы:** Выбор оптимального значения 'p' может быть сложным. |
| Поиск по лучу (Beam Search) | Исследует несколько гипотез (последовательностей слов) на каждом шаге и сохраняет 'k' наиболее вероятных, где 'k' — ширина луча. | **Плюсы:** Даёт более надёжные результаты, чем жадный поиск. <br> **Минусы:** Может не хватать разнообразия и приводить к шаблонным ответам. |
| Топ-k сэмплирование (Top-k Sampling) | Случайно выбирает следующее слово из 'k' слов с наибольшими вероятностями. | **Плюсы:** Вносит случайность, увеличивая разнообразие текста. <br> **Минусы:** Случайный выбор может иногда приводить к менее связным результатам. |
| Нормализация длины (Length Normalization) | Предотвращает предпочтение более коротких последовательностей путём деления логарифмических вероятностей на длину последовательности, возведённую в некоторую степень. | **Плюсы:** Делает более длинные и потенциально более информативные последовательности более вероятными. <br> **Минусы:** Настройка коэффициента нормализации может быть сложной. |
| Стохастический поиск по лучу (Stochastic Beam Search) | Вносит случайность в процесс выбора 'k' гипотез в поиске по лучу. | **Плюсы:** Увеличивает разнообразие генерируемого текста. <br> **Минусы:** Баланс между разнообразием и качеством может быть сложным для управления. |
| Декодирование с минимальным байесовским риском (MBR) | Выбирает гипотезу (из множества), которая минимизирует ожидаемые потери согласно функции потерь. | **Плюсы:** Оптимизирует вывод согласно определённой функции потерь. <br> **Минусы:** Вычислительно сложнее и требует хорошей функции потерь. |

Ссылки на документацию:
- [справка для `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [справка для `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Документация Huggingface [по стратегиям генерации](https://huggingface.co/docs/transformers/generation_strategies)

In [71]:
prompt = "SSH service is"

inputs = tokenizer([prompt], return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=10, do_sample=True, temperature=0.8, top_k=50, eos_token_id=model.config.eos_token_id)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


['SSH service is typically run with:\nA. Root privileges\nB']

Chat template

In [73]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to("cpu")
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

In [82]:
messages = [
  {"role": "system", "content": "Всегда отвечай на русском языке"},
    
  {"role": "user", "content": "Привет!"},

  {"role": "assistant", "content": "Чем могу тебе помочь?"},

  {"role": "user", "content": "Напиши в двух предложениях, что такое ssh сервер"},

]

In [83]:
tokenizer.apply_chat_template(messages, tokenize=False)

'<|im_start|>system\nВсегда отвечай на русском языке<|im_end|>\n<|im_start|>user\nПривет!<|im_end|>\n<|im_start|>assistant\nЧем могу тебе помочь?<|im_end|>\n<|im_start|>user\nНапиши в двух предложениях, что такое ssh сервер<|im_end|>\n'

In [85]:
context = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
outputs = model.generate(context, max_new_tokens=128)
print(tokenizer.decode(outputs[0]))

<|im_start|>system
Всегда отвечай на русском языке<|im_end|>
<|im_start|>user
Привет!<|im_end|>
<|im_start|>assistant
Чем могу тебе помочь?<|im_end|>
<|im_start|>user
Напиши в двух предложениях, что такое ssh сервер<|im_end|>
<|im_start|>assistant
SSH (Secure Shell) — это специальная версия протокола TCP/UDP, которая позволяет пользователям перенаправлять и выполнять команды через компьютеры друг друга. Это обеспечивает безопасность передачи данных между двумя компьютерами, особенно в контексте удаленной работы или разговоров. Важно отметить, что SSH использует протокол SSH, который обычно используется для доступа к сети по порту 22.<|im_end|>


Дообучение (finetuning) модели